# 02 — Model Evaluation

Systematic post-training analysis of the YOLO-OBB boat detection model.

All metric computation is delegated to `src.vessels_detect.vessels_detect.vessels_detect.evaluation.metrics`.
This notebook contains only configuration constants and visualisation code.

| § | Description |
|---|-------------|
| 1 | Setup & constants |
| 2 | Model loading & formal val() metrics |
| 3 | Per-class performance table |
| 4 | Normalised confusion matrix |
| 5 | Dataset statistics |
| 6 | OBB geometric statistics |
| 7 | PR curve |
| 8 | TP / FP / FN detection crops |

In [1]:
%env OPENCV_LOG_LEVEL=ERROR
# ── Standard library ──────────────────────────────────────────────────────
import sys, logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

# ── Third-party ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import cv2

# ── Project ───────────────────────────────────────────────────────────────
from src.vessels_detect.utils.config import load_config
from src.vessels_detect.evaluation.metrics import (
    compute_per_class_metrics,
    compute_confusion_matrix,
    build_size_dataframe,
    build_distribution_dataframe,
    parse_label_file,
    match_detections,
    compute_crop_metrics,
    build_crop_summary,
    # § 6 — Systematic Counting Bias
    calculate_counting_bias,
    build_bias_dataframe,
)

logging.getLogger("ultralytics").setLevel(logging.ERROR)

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "legend.framealpha": 0.7,
    "legend.fontsize": 9,
})
print("Imports OK.")

env: OPENCV_LOG_LEVEL=ERROR
Imports OK.


## 1. Constants

Change these to adapt to a different run or dataset.

In [2]:
MODEL_PATH    = Path("../boat_obb_v1/train/weights/best.pt")
DATA_PATH     = Path("../data/dataset.yaml")
PROCESSED_DIR = Path("../data/processed")
LABELS_ROOT   = PROCESSED_DIR / "labels"
IMAGES_DIR    = PROCESSED_DIR / "images"
LABELS_DIR    = PROCESSED_DIR / "labels"

SPLITS        = ["train", "val", "test"]

FINAL_CONF    = 0.10
FINAL_IOU     = 0.30
IMG_SIZE      = 640      # must match training imgsz
TILE_SIZE     = 640      # tile size used in preprocessing

print(f"Model  : {MODEL_PATH}")
print(f"Data   : {DATA_PATH}")
print(f"Conf   : {FINAL_CONF}  |  IoU : {FINAL_IOU}")

Model  : ../boat_obb_v1/train/weights/best.pt
Data   : ../data/dataset.yaml
Conf   : 0.1  |  IoU : 0.3


## 2. Model Loading

In [3]:
from ultralytics import YOLO

model       = YOLO(MODEL_PATH)
CLASS_NAMES = model.names
print("Classes:", CLASS_NAMES)

Classes: {0: 'Pirogue', 1: 'Double_hulled_Pirogue', 2: 'Small_Motorboat', 3: 'Medium_Motorboat', 4: 'Large_Motorboat', 5: 'Sailing_Boat'}


## 3. Formal Evaluation — model.val()

In [4]:
metrics = model.val(
    data=str(DATA_PATH),
    split="test",
    imgsz=IMG_SIZE,
    conf=FINAL_CONF,
    iou=FINAL_IOU,
    save_json=False,
    plots=False,
    verbose=False,
)

print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

Ultralytics 8.4.1 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4080, 15937MiB)
YOLO26m-obb summary (fused): 142 layers, 21,202,529 parameters, 0 gradients, 71.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6070.6±3512.5 MB/s, size: 1298.0 KB)
val: Scanning /home/thomas/Documents/test/pleiades-boat-detection/data/processed/labels/test.cache... 6411 images, 6253 backgrounds, 38 corrupt: 100% ━━━━━━━━━━━━ 6411/6411 1.8Git/s 0.0s
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_MS-FS_ORT_PWOI_000373512_2_2_F_1_RGB_R1C1_15104_2560.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.092055]
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_MS-FS_ORT_PWOI_000373512_2_2_F_1_RGB_R1C1_15104_2816.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.0230551]
val: /home/thomas/Documen

## 4. Per-Class Performance Table

In [5]:
df_perf = compute_per_class_metrics(metrics, CLASS_NAMES, LABELS_ROOT, test_split="test")
display(df_perf)

,Class,Support,Precision,Recall,F1-score,mAP50,mAP50-95
0,Small_Motorboat,47,0.4000,0.8333,0.5405,0.6221,0.4846
1,Double_hulled_Pirogue,284,0.4387,0.5763,0.4982,0.5672,0.4139
2,Sailing_Boat,17,1.0000,0.2000,0.3333,0.6000,0.3007
3,Pirogue,158,0.0723,0.5495,0.1277,0.1505,0.0922
4,Medium_Motorboat,0,NaN,NaN,NaN,NaN,NaN
5,Large_Motorboat,0,NaN,NaN,NaN,NaN,NaN
6,**Global**,506,0.4777,0.5398,0.5069,0.4850,0.3007


## 5. Normalised Confusion Matrix

In [6]:
metrics_cm = model.val(
    data=str(DATA_PATH), split="test",
    imgsz=IMG_SIZE, conf=FINAL_CONF, iou=FINAL_IOU,
    plots=True, save_json=False, verbose=False,
)

cm_norm, labels = compute_confusion_matrix(metrics_cm, CLASS_NAMES)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm_norm, vmin=0, vmax=1, cmap="Blues")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ticks = np.arange(len(labels))
ax.set_xticks(ticks); ax.set_xticklabels(labels, rotation=40, ha="right")
ax.set_yticks(ticks); ax.set_yticklabels(labels)

thresh = cm_norm.max() / 2.0
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        v = cm_norm[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=8, color="white" if v > thresh else "black")

ax.set_ylabel("True label"); ax.set_xlabel("Predicted label")
ax.set_title(f"Normalised Confusion Matrix (conf={FINAL_CONF}, IoU={FINAL_IOU})",
             fontweight="bold")
plt.tight_layout()
plt.show()

Ultralytics 8.4.1 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4080, 15937MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4879.9±2233.4 MB/s, size: 920.7 KB)
val: Scanning /home/thomas/Documents/test/pleiades-boat-detection/data/processed/labels/test.cache... 6411 images, 6253 backgrounds, 38 corrupt: 100% ━━━━━━━━━━━━ 6411/6411 3.8Git/s 0.0s
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_MS-FS_ORT_PWOI_000373512_2_2_F_1_RGB_R1C1_15104_2560.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.092055]
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_MS-FS_ORT_PWOI_000373512_2_2_F_1_RGB_R1C1_15104_2816.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.0230551]
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_M

<Figure size 1350x1200 with 2 Axes>

## 9. Detection Crop Export — TP / FP / FN

Crops are saved to `crops/<TruePositive|FalsePositive|FalseNegative>/<class>/`.
A summary CSV is written to `crops/summary.csv`.

In [11]:
IOU_MATCH  = FINAL_IOU
CROP_PAD   = 80
CROPS_ROOT = Path("../crops")
SPLIT      = "test"

OBB_COLOR     = (0, 0, 255)
OBB_THICKNESS = 2

for sub in ("TruePositive", "FalsePositive", "FalseNegative"):
    (CROPS_ROOT / sub).mkdir(parents=True, exist_ok=True)


def crop_obb(img_bgr, pts_px, pad=CROP_PAD):
    h, w = img_bgr.shape[:2]
    x1 = max(0,     int(pts_px[:, 0].min()) - pad)
    y1 = max(0,     int(pts_px[:, 1].min()) - pad)
    x2 = min(w - 1, int(pts_px[:, 0].max()) + pad)
    y2 = min(h - 1, int(pts_px[:, 1].max()) + pad)
    return img_bgr[y1:y2, x1:x2].copy(), (x1, y1)

def draw_obb(img, pts_px, origin=(0, 0)):
    shifted = pts_px.copy()
    shifted[:, 0] -= origin[0]; shifted[:, 1] -= origin[1]
    cv2.polylines(img, [shifted.astype(np.int32).reshape(-1, 1, 2)],
                  True, OBB_COLOR, OBB_THICKNESS)
    return img


summary_rows = []
tp_count = fp_count = fn_count = 0

for img_path in sorted((IMAGES_DIR / SPLIT).glob("*.tif")):
    import rasterio as _rio
    with _rio.open(img_path) as ds:
        rgb_data = ds.read()
    img_bgr = cv2.cvtColor(rgb_data[:3].transpose(1, 2, 0), cv2.COLOR_RGB2BGR)
    h, w = img_bgr.shape[:2]
    stem  = img_path.stem

    lbl = LABELS_ROOT / SPLIT / (stem + ".txt")
    gt  = parse_label_file(lbl, w, h)

    res   = model.predict(str(img_path), imgsz=IMG_SIZE,
                          conf=FINAL_CONF, iou=FINAL_IOU, verbose=False)[0]
    preds = []
    if res.obb is not None and len(res.obb):
        for pts, cid, cf in zip(
            res.obb.xyxyxyxy.cpu().numpy(),
            res.obb.cls.cpu().numpy().astype(int),
            res.obb.conf.cpu().numpy(),
        ):
            preds.append((int(cid), pts.astype(np.float32), float(cf)))

    m_pred, m_gt = match_detections(gt, preds, IOU_MATCH)

    # TPs
    for pi, (pr_cid, pr_pts, pr_cf) in enumerate(preds):
        if m_pred[pi]:
            crop, origin = crop_obb(img_bgr, pr_pts)
            if crop.size:
                crop = draw_obb(crop, pr_pts, origin)
                out  = CROPS_ROOT / "TruePositive" / CLASS_NAMES[pr_cid]
                out.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(out / f"{stem}__tp__{tp_count:04d}.png"), crop)
                summary_rows.append({"image": stem, "category": "TruePositive",
                                     "class": CLASS_NAMES[pr_cid], "index": tp_count})
                tp_count += 1

    # FPs
    for pi, (pr_cid, pr_pts, pr_cf) in enumerate(preds):
        if not m_pred[pi]:
            out = CROPS_ROOT / "FalsePositive" / CLASS_NAMES[pr_cid]
            out.mkdir(parents=True, exist_ok=True)
            canvas = img_bgr.copy()
            draw_obb(canvas, pr_pts)
            cv2.imwrite(str(out / f"{stem}__fp__{fp_count:04d}.png"), canvas)
            summary_rows.append({"image": stem, "category": "FalsePositive",
                                 "class": CLASS_NAMES[pr_cid], "index": fp_count})
            fp_count += 1

    # FNs
    for gi, (gt_cid, gt_pts) in enumerate(gt):
        if not m_gt[gi]:
            crop, _ = crop_obb(img_bgr, gt_pts, pad=CROP_PAD)
            if crop.size:
                out = CROPS_ROOT / "FalseNegative" / CLASS_NAMES[gt_cid]
                out.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(out / f"{stem}__fn__{fn_count:04d}.png"), crop)
                summary_rows.append({"image": stem, "category": "FalseNegative",
                                     "class": CLASS_NAMES[gt_cid], "index": fn_count})
                fn_count += 1

df_crops = build_crop_summary(summary_rows)
if not df_crops.empty:
    df_crops.to_csv(CROPS_ROOT / "summary.csv", index=False)

p, r, f1 = compute_crop_metrics(tp_count, fp_count, fn_count)
print(f"TP: {tp_count}  FP: {fp_count}  FN: {fn_count}")
print(f"Precision: {p:.4f}  Recall: {r:.4f}  F1: {f1:.4f}")
print(f"Crops saved to: {CROPS_ROOT.resolve()}")
display(df_crops)

TP: 323  FP: 1083  FN: 183
Precision: 0.2297  Recall: 0.6383  F1: 0.3379
Crops saved to: /home/thomas/Documents/test/pleiades-boat-detection/crops


,category,class,count
0,FalseNegative,Double_hulled_Pirogue,95
1,FalseNegative,Pirogue,66
2,FalseNegative,Sailing_Boat,7
3,FalseNegative,Small_Motorboat,15
4,FalsePositive,Double_hulled_Pirogue,132
5,FalsePositive,Medium_Motorboat,1
6,FalsePositive,Pirogue,922
7,FalsePositive,Sailing_Boat,2
8,FalsePositive,Small_Motorboat,26
9,TruePositive,Double_hulled_Pirogue,166
